In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Morgan State University × Google Cloud — Researcher Day

> 👋 **Welcome!** In the next hour you will run a Colab Enterprise notebook on your own **NVIDIA H100 GPU (80 GB)**, query real federal datasets with **BigQuery**, mount the lab's **Cloud Storage** bucket with **Cloud Storage FUSE**, run **Gemma 3** from **Vertex AI Model Garden** on your GPU, optionally deploy it to a production **Vertex AI endpoint**, and compare with Google's managed **Gemini** model. 🚀

## 1. Setup

You should already be connected to the runtime you created (see the README); run every cell from here with the run button or Shift+Enter.


### 1.1 — Project + lab parameters

In [ ]:
import os

# Colab Enterprise sets GOOGLE_CLOUD_PROJECT to the project that owns your runtime.
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "")
# If the line below prints an empty project, paste the sandbox project id in the next line and re-run:
# PROJECT_ID = "the-sandbox-project-id"

REGION = "us-east5"              # all lab resources live here (deepest H100 availability)
RUN_OPTIONAL_DEPLOY = False      # flip to True only if you do Section 6 (costs about $11/hr while deployed)

print(f"PROJECT_ID          = {PROJECT_ID!r}")
print(f"REGION              = {REGION!r}")
print(f"RUN_OPTIONAL_DEPLOY = {RUN_OPTIONAL_DEPLOY}")
assert PROJECT_ID, "PROJECT_ID is empty — paste the sandbox project id above and re-run."


### 1.2 — Install the two packages the runtime image doesn't already have (30 seconds; safe to re-run)

We deliberately do NOT upgrade preinstalled libraries — that destabilizes a live kernel.

In [ ]:
%pip install --quiet "google-genai>=2.8.0" "db-dtypes>=1.7.0"
print("packages ready")


### 1.3 — Who is this notebook acting as? (your credentials, not a robot's)

In [ ]:
import google.auth

credentials, adc_project = google.auth.default()
print("ADC project :", adc_project)
print("Identity    :", getattr(credentials, "service_account_email", "end-user credentials (you)"))
!gcloud config list account --format "value(core.account)"


### 1.4 — Enable the APIs this lab uses (idempotent; 30 seconds on first run)

In [ ]:
!gcloud services enable aiplatform.googleapis.com dataform.googleapis.com compute.googleapis.com \
    bigquery.googleapis.com bigquerystorage.googleapis.com storage.googleapis.com \
    --project={PROJECT_ID}
print("APIs enabled")


### 1.5 — Confirm the H100 is attached

In [ ]:
!nvidia-smi


### 1.6 — Can PyTorch see it?

In [ ]:
import torch

print("torch version      :", torch.__version__)
print("CUDA available     :", torch.cuda.is_available())
print("GPU                :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
assert torch.cuda.is_available(), "No GPU visible — are you connected to your msu-h100 runtime (not the default CPU one)?"


## 2. CPU vs GPU

Same math, two processors: a 4096×4096 matrix multiplication, ten times on the CPU, ten times on the H100. This is the primitive underneath every neural network you'll touch today.


### 2.1 — CPU vs H100 on a 4096×4096 matmul (×10)

In [ ]:
import time

import torch

N, REPS = 4096, 10

a_cpu = torch.randn(N, N)
b_cpu = torch.randn(N, N)
t0 = time.perf_counter()
for _ in range(REPS):
    _ = a_cpu @ b_cpu
cpu_s = time.perf_counter() - t0

a_gpu = a_cpu.to("cuda")
b_gpu = b_cpu.to("cuda")
_ = a_gpu @ b_gpu                      # warm-up (CUDA init + kernel load)
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(REPS):
    _ = a_gpu @ b_gpu
torch.cuda.synchronize()
gpu_s = time.perf_counter() - t0

print(f"CPU  : {cpu_s:6.2f} s")
print(f"H100 : {gpu_s:6.2f} s")
print(f"→ the H100 is about {cpu_s / gpu_s:,.0f}x faster on this workload")


### 2.2 — GPU memory after the test

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

# free GPU memory before section 5 loads a language model
del a_gpu, b_gpu
import torch; torch.cuda.empty_cache()
print("GPU memory released")


## 3. BigQuery

**BigQuery** is Google Cloud's serverless data warehouse. The [public dataset program](https://console.cloud.google.com/bigquery?p=bigquery-public-data) hosts hundreds of real datasets — you pay only for what a query scans, and **the first 1 TB/month is free**.

We'll use two that match Morgan State research areas:
- **NHTSA Fatality Analysis Reporting System** — every fatal crash in the U.S. → [open dataset](https://console.cloud.google.com/bigquery?p=bigquery-public-data&d=nhtsa_traffic_fatalities&page=dataset) *(transportation safety)*
- **Census American Community Survey** — demographics & income by county → [open dataset](https://console.cloud.google.com/bigquery?p=bigquery-public-data&d=census_bureau_acs&page=dataset) *(urban & equity research)*

The same SQL runs interactively in **BigQuery Studio**: https://console.cloud.google.com/bigquery


### 3.1 — Wire the %%bigquery magic to your project

In [ ]:
try:
    import bigquery_magics
    get_ipython().run_line_magic("load_ext", "bigquery_magics")
    bigquery_magics.context.project = PROJECT_ID
    print("bigquery_magics bound to", PROJECT_ID)
except ImportError:
    get_ipython().run_line_magic("load_ext", "google.cloud.bigquery")
    from google.cloud.bigquery import magics
    magics.context.project = PROJECT_ID
    print("google.cloud.bigquery magics bound to", PROJECT_ID)


### 3.2 — Maryland fatal crashes by month (FARS 2016 snapshot)

In [ ]:
%%bigquery md_crashes
SELECT
  month_of_crash,
  COUNT(*)                  AS fatal_crashes,
  SUM(number_of_fatalities) AS deaths
FROM `bigquery-public-data.nhtsa_traffic_fatalities.accident_2016`
WHERE state_name = "Maryland"
GROUP BY month_of_crash
ORDER BY month_of_crash


### 3.3 — Baltimore vs neighboring counties: income & education (ACS 2020 5-yr)

In [ ]:
%%bigquery balt_acs
SELECT
  CASE geo_id
    WHEN "24510" THEN "Baltimore City"
    WHEN "24005" THEN "Baltimore County"
    WHEN "24003" THEN "Anne Arundel"
    WHEN "24027" THEN "Howard"
    WHEN "24031" THEN "Montgomery"
  END                               AS county,
  total_pop,
  median_income,
  bachelors_degree_or_higher_25_64  AS bachelors_plus_25_64
FROM `bigquery-public-data.census_bureau_acs.county_2020_5yr`
WHERE geo_id IN ("24510", "24005", "24003", "24027", "24031")
ORDER BY median_income


### 3.4 — Chart the results

Both results are now pandas DataFrames in this kernel.

In [ ]:
import matplotlib.pyplot as plt

print(md_crashes)
print(balt_acs)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(md_crashes["month_of_crash"], md_crashes["fatal_crashes"], color="#4285F4")
ax.set_xlabel("Month")
ax.set_ylabel("Fatal crashes")
ax.set_title("Maryland fatal crashes by month — FARS 2016")
ax.set_xticks(range(1, 13))
plt.tight_layout()
plt.show()


> **Also:** Morgan State itself is in the federal IPEDS dataset. Paste this into [BigQuery Studio](https://console.cloud.google.com/bigquery):
> ```sql
> SELECT instnm, city, stabbr, hbcu
> FROM `bigquery-public-data.nces_ipeds.hd2020`
> WHERE instnm LIKE "%Morgan State%" OR (hbcu = 1 AND stabbr = "MD")
> ```


## 4. Cloud Storage + FUSE

**Cloud Storage (GCS)** is where research data lives between sessions (your runtime's disk is wiped when it's deleted — buckets persist and cost $0.02/GB-month). **Cloud Storage FUSE** mounts a bucket into the filesystem so anything — shell, pandas, a training loop — can read/write it like local files.

The sandbox project shares **one** lab bucket; your work lives under **your own folder** inside it. You will create your folder, mount the bucket, and save today's query results there.


### 4.1 — The lab's shared bucket + YOUR folder in it

In [ ]:
import subprocess

BUCKET = f"{PROJECT_ID}-researcher-day"          # one bucket for the whole room
account = subprocess.run(["gcloud", "config", "get-value", "account"],
                         capture_output=True, text=True).stdout.strip()
RESEARCHER = account.split("@")[0].replace(".", "-").lower() or "researcher"
print(f"you are {account} — your folder: {RESEARCHER}/")

exists = subprocess.run(
    ["gcloud", "storage", "buckets", "describe", f"gs://{BUCKET}", "--format=value(name)"],
    capture_output=True, text=True,
).returncode == 0
if exists:
    print(f"bucket gs://{BUCKET} already exists (shared)")
else:
    !gcloud storage buckets create gs://{BUCKET} --location={REGION} --uniform-bucket-level-access --project={PROJECT_ID}
    print(f"created gs://{BUCKET}")

print(f"console: https://console.cloud.google.com/storage/browser/{BUCKET}?project={PROJECT_ID}")


### 4.2 — Save the BigQuery results locally first

In [ ]:
md_crashes.to_csv("md_crashes_2016.csv", index=False)
balt_acs.to_csv("baltimore_acs_2020.csv", index=False)
print("wrote md_crashes_2016.csv and baltimore_acs_2020.csv")


### 4.3 — Install Cloud Storage FUSE (official apt repo; 30 seconds)

In [ ]:
%%bash
set -e
if command -v gcsfuse >/dev/null 2>&1; then
  echo "gcsfuse already installed: $(gcsfuse --version)"
  exit 0
fi
export GCSFUSE_REPO="gcsfuse-$(lsb_release -c -s)"
curl -fsSL https://packages.cloud.google.com/apt/doc/apt-key.gpg \
  | gpg --dearmor -o /usr/share/keyrings/cloud.google.asc 2>/dev/null \
  || curl -fsSL https://packages.cloud.google.com/apt/doc/apt-key.gpg -o /usr/share/keyrings/cloud.google.asc
echo "deb [signed-by=/usr/share/keyrings/cloud.google.asc] https://packages.cloud.google.com/apt $GCSFUSE_REPO main" \
  > /etc/apt/sources.list.d/gcsfuse.list
apt-get update -qq
apt-get install -y -qq gcsfuse
gcsfuse --version


### 4.4 — Mount the bucket and copy the CSVs into YOUR folder

In [ ]:
import os
import shutil
import subprocess

MOUNT = os.path.expanduser("~/gcs-mount")
os.makedirs(MOUNT, exist_ok=True)

FUSE_OK = os.path.exists("/dev/fuse")
if FUSE_OK:
    subprocess.run(["fusermount", "-u", MOUNT], capture_output=True)   # clear stale mount on re-run
    r = subprocess.run(["gcsfuse", "--implicit-dirs", BUCKET, MOUNT], capture_output=True, text=True)
    print(r.stdout)
    print(r.stderr)
    FUSE_OK = r.returncode == 0

if FUSE_OK:
    MY_DIR = os.path.join(MOUNT, RESEARCHER)
    os.makedirs(MY_DIR, exist_ok=True)
    shutil.copy("md_crashes_2016.csv", MY_DIR)
    shutil.copy("baltimore_acs_2020.csv", MY_DIR)
    print(f"bucket mounted at {MOUNT}; your folder works like a local directory:")
    print(os.listdir(MY_DIR))
else:
    MY_DIR = "."
    print("FUSE unavailable on this runtime; using gcloud storage cp instead (same result):")
    !gcloud storage cp md_crashes_2016.csv baltimore_acs_2020.csv gs://{BUCKET}/{RESEARCHER}/


### 4.5 — Proof: your objects are really in Cloud Storage

In [ ]:
!gcloud storage ls -l gs://{BUCKET}/{RESEARCHER}/
print(f"console: https://console.cloud.google.com/storage/browser/{BUCKET}/{RESEARCHER}?project={PROJECT_ID}")


## 5. Model Garden + Gemma 3 on your H100

**[Vertex AI Model Garden](https://console.cloud.google.com/vertex-ai/model-garden)** is the catalog of 200+ models you can use on Google Cloud — Google's own (Gemini, **Gemma**, including Gemma 4), partner models (Claude, Llama, Mistral…), and task models. Each card shows hardware configs Google has *verified* — from L4s up to `a3-highgpu-1g` with the same **H100** you're sitting on (and the new G4 / RTX PRO 6000).

Open the [**Gemma 3 card**](https://console.cloud.google.com/vertex-ai/publishers/google/model-garden/gemma3) → see the **Deploy** options. The H100 config in that list is the production path you can try in Section 6.

First, the SDK view of the same catalog:


### 5.1 — The Model Garden catalog, from Python

In [ ]:
import vertexai
from vertexai import model_garden

vertexai.init(project=PROJECT_ID, location=REGION)

gemma_models = model_garden.list_deployable_models(model_filter="gemma")
print(f"{len(gemma_models)} deployable Gemma-family models in Model Garden:")
for m in gemma_models:
    print("  ", m)


### 5.2 — What hardware has Google verified for gemma-3-4b-it? (look for the H100 config)

In [ ]:
from vertexai import model_garden

deploy_options = model_garden.OpenModel("google/gemma3@gemma-3-4b-it").list_deploy_options()
for i, opt in enumerate(deploy_options):
    spec = opt.dedicated_resources.machine_spec
    print(f"option {i}: machine={spec.machine_type}  accelerator={spec.accelerator_type.name} x{spec.accelerator_count}")


### Run Gemma 3 locally on this runtime's H100

For interactive work you don't need an endpoint at all: the weights fit right on your runtime's GPU. We'll use **[Ollama](https://ollama.com/library/gemma3)**, an open-source model server (handy on your laptop later, too) — it downloads **Gemma 3 4B** (3 GB, no account needed) and serves it on the H100. Section 8 lists the command for the 27B variant.

The first chat call loads the model into GPU memory and takes 30 to 90 seconds. Later calls are fast.


### 5.3 — Install Ollama (30 seconds; zstd first — the installer's archives need it)

In [ ]:
%%bash
set -e
if command -v ollama >/dev/null 2>&1; then echo "ollama already installed"; exit 0; fi
apt-get install -y -qq zstd
curl -fsSL https://ollama.com/install.sh | sh


### 5.4 — Start the Ollama server and wait until it answers

In [ ]:
%%bash
nohup ollama serve > /tmp/ollama.log 2>&1 &
for i in $(seq 1 30); do
  if curl -s http://127.0.0.1:11434/api/version; then echo; echo "ollama server is up"; exit 0; fi
  sleep 1
done
echo "server did not come up — full log:"; cat /tmp/ollama.log; exit 1


### 5.5 — Pull Gemma 3 4B onto your GPU (3 GB, 1-3 minutes on GCP's network)

In [ ]:
!ollama pull gemma3:4b
!ollama list


### 5.6 — First request to the model

In [ ]:
import json

import requests

OLLAMA_URL = "http://127.0.0.1:11434/api/chat"

request_payload = {
    "model": "gemma3:4b",
    "messages": [{"role": "user", "content": "In two sentences: why would a university researcher want a GPU in the cloud instead of under their desk?"}],
    "stream": False,
}
print("=== FULL REQUEST PAYLOAD ===")
print(json.dumps(request_payload, indent=2))

resp = requests.post(OLLAMA_URL, json=request_payload, timeout=300)
resp.raise_for_status()
response_payload = resp.json()

print("=== FULL RESPONSE PAYLOAD ===")
print(json.dumps(response_payload, indent=2))

print("=== GEMMA SAYS ===")
print(response_payload["message"]["content"])


### 5.7 — GPU memory now held by the model server

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv


### BigQuery to Gemma to Cloud Storage, one pipeline

Hand the model your Section-3 query results and have it draft a research brief — then archive the brief in your bucket. Three products, one cell each.

> **Exercise:** read the brief critically — at least one of its claims is **not supported by the data you gave it** (small local models overreach; e.g., watch for county-level crash claims when DATA 1 has no county column). Spotting it is the point: LLM drafts always need a researcher's review. Section 7 shows how a frontier model handles the same task.


### 5.8 — Gemma writes a Maryland road-safety brief from YOUR query results

In [ ]:
import json

import requests

brief_prompt = f"""You are drafting notes for Morgan State University transportation and urban-policy researchers.

DATA 1 — Maryland fatal crashes by month, FARS 2016 (CSV):
{md_crashes.to_csv(index=False)}

DATA 2 — Income & education, Baltimore City vs neighboring Maryland counties, ACS 2020 5-year (CSV):
{balt_acs.to_csv(index=False)}

Write a research brief with: (1) a 2-sentence summary, (2) three bullet observations grounded in the numbers,
(3) one observation connecting road safety investment to the income gap visible in DATA 2,
(4) two follow-up research questions suitable for a grad seminar.
"""

request_payload = {"model": "gemma3:4b", "messages": [{"role": "user", "content": brief_prompt}], "stream": False}
print("=== FULL REQUEST PAYLOAD ===")
print(json.dumps(request_payload, indent=2))

resp = requests.post(OLLAMA_URL, json=request_payload, timeout=600)
resp.raise_for_status()
response_payload = resp.json()
print("=== FULL RESPONSE PAYLOAD ===")
print(json.dumps(response_payload, indent=2))

gemma_brief = response_payload["message"]["content"]
print("=== THE BRIEF ===")
print(gemma_brief)


### 5.9 — Archive the brief in YOUR folder (via the FUSE mount if live)

In [ ]:
import os

brief_path = os.path.join(MY_DIR, "brief.md")
with open(brief_path, "w") as f:
    f.write("# Maryland road-safety brief — drafted by gemma3:4b on an NVIDIA H100\n\n")
    f.write(gemma_brief)
print("wrote", brief_path)

if MY_DIR == ".":
    !gcloud storage cp brief.md gs://{BUCKET}/{RESEARCHER}/

!gcloud storage ls -l gs://{BUCKET}/{RESEARCHER}/
print(f"console: https://console.cloud.google.com/storage/browser/{BUCKET}/{RESEARCHER}?project={PROJECT_ID}")


## 6. *Optional* — deploy Gemma to a production endpoint

Local Ollama is perfect for exploration. When your model needs to serve an **application** — a web app, a data pipeline, your lab's API — you deploy it from Model Garden to a **Vertex AI endpoint**: managed, autoscaled, authenticated, on hardware Google has verified (here: `a3-highgpu-1g` with **another NVIDIA H100** — the *same* `gemma-3-4b-it` you're running locally, now production-grade).

Costs about $11 per hour while deployed; the last cell un-deploys. Takes 5 to 15 minutes to go live.

> **Quota note:** the sandbox project's `Custom model serving Nvidia H100 GPUs` quota (16 by default) is shared by the whole room. If the deploy fails on quota, the shared slots are in use — run the cleanup cell when you finish so others get a slot, or return to this section after the event.

To run this section: set `RUN_OPTIONAL_DEPLOY = True` in cell 1.1, re-run it, then run the cells below.


### 6.1 — Deploy gemma-3-4b-it to a Vertex endpoint on 1× H100 (5-15 minutes)

In [ ]:
if not RUN_OPTIONAL_DEPLOY:
    print("skipped — set RUN_OPTIONAL_DEPLOY = True in cell 1.1 to run the production path")
else:
    import vertexai
    from vertexai import model_garden

    vertexai.init(project=PROJECT_ID, location=REGION)
    open_model = model_garden.OpenModel("google/gemma3@gemma-3-4b-it")
    # Explicit machine args pin the verified H100 config (the catalog default moves with new GPUs).
    endpoint = open_model.deploy(
        machine_type="a3-highgpu-1g",
        accelerator_type="NVIDIA_H100_80GB",
        accelerator_count=1,
        accept_eula=True,
    )
    print("deployed:", endpoint.resource_name)
    print(f"console: https://console.cloud.google.com/vertex-ai/online-prediction/endpoints?project={PROJECT_ID}")


### 6.2 — Call your endpoint (same Gemma, now behind a managed, authenticated API)

In [ ]:
if not RUN_OPTIONAL_DEPLOY:
    print("skipped")
else:
    import json

    request_instances = [{
        "@requestFormat": "chatCompletions",
        "messages": [{"role": "user", "content": "One sentence: what is Morgan State University known for?"}],
        "max_tokens": 100,
    }]
    print("=== FULL REQUEST PAYLOAD ===")
    print(json.dumps(request_instances, indent=2))

    prediction = endpoint.predict(instances=request_instances)
    print("=== FULL RESPONSE PAYLOAD ===")
    print(json.dumps(prediction.predictions, indent=2, default=str))


### 6.3 — Clean up the endpoint (stops the $11/hr the moment you're done)

In [ ]:
if not RUN_OPTIONAL_DEPLOY:
    print("skipped")
else:
    from google.cloud import aiplatform

    deployed_model_names = [m.model for m in endpoint.list_models()]
    endpoint.undeploy_all()
    endpoint.delete()
    for model_name in deployed_model_names:
        aiplatform.Model(model_name).delete()
    print("endpoint + model deleted — verify the list is empty:")
    !gcloud ai endpoints list --region={REGION} --project={PROJECT_ID}


## 7. Bonus — the same task on managed Gemini

You just ran an open model on hardware you control. The other end of the spectrum: **Gemini on Vertex AI** — no GPU, no server, no deploy; you call Google's frontier model directly and pay per token. Same brief task, for comparison:

Prompt it interactively in **Vertex AI Studio**: https://console.cloud.google.com/vertex-ai/studio


### 7.1 — Same research-brief prompt → gemini-3.1-pro-preview (managed, zero infrastructure)

In [ ]:
import json

from google import genai
from google.genai import types

client = genai.Client(vertexai=True, project=PROJECT_ID, location="global")
GEMINI_MODEL = "gemini-3.1-pro-preview"

request_payload = {"model": GEMINI_MODEL, "contents": brief_prompt, "config": {"temperature": 0.3}}
print("=== FULL REQUEST PAYLOAD ===")
print(json.dumps(request_payload, indent=2))

response = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=brief_prompt,
    config=types.GenerateContentConfig(temperature=0.3),
)
print("=== FULL RESPONSE PAYLOAD ===")
print(response.model_dump_json(indent=2, exclude_none=True))

print("=== GEMINI'S BRIEF ===")
print(response.text)


**Compare the two briefs.** Gemma 3 4B ran on *your* H100 with full data control; Gemini ran as a managed service with frontier quality and zero setup. Real research stacks use both — open models where control/cost matter, managed APIs where capability/speed matter.


## 8. Cleanup and next steps

**What keeps costing money after you close this tab:**
| Resource | Cost while alive | How to stop it |
|---|---|---|
| Your H100 runtime | about $11/hr | Auto-stops after 180 idle minutes — **delete it when you leave**: [Runtimes](https://console.cloud.google.com/vertex-ai/colab/runtimes) → ⋮ → Delete |
| Section 6 endpoint (if you deployed **and skipped cell 6.3**) | about $11/hr | [Endpoints](https://console.cloud.google.com/vertex-ai/online-prediction/endpoints) → undeploy + delete |
| The lab bucket | $0.02/GB-month | Keep it; that is where research data should live |
| BigQuery queries | First 1 TB/month free | Nothing to do |

Your notebook file itself is free — it lives in Colab Enterprise, not on the runtime.

Today's spend appears the next morning: [Billing reports](https://console.cloud.google.com/billing) → filter label `goog-vertex-ai-product: colab-enterprise`.


### 8.1 — Your personalized link wall (bookmark these)

In [ ]:
links = {
    "Colab Enterprise — notebooks":   f"https://console.cloud.google.com/vertex-ai/colab/notebooks?project={PROJECT_ID}",
    "Colab Enterprise — runtimes":    f"https://console.cloud.google.com/vertex-ai/colab/runtimes?project={PROJECT_ID}",
    "Runtime templates":              f"https://console.cloud.google.com/vertex-ai/colab/runtime-templates?project={PROJECT_ID}",
    "Notebook gallery (templates)":   f"https://console.cloud.google.com/vertex-ai/colab/notebook-gallery?project={PROJECT_ID}",
    "BigQuery Studio":                f"https://console.cloud.google.com/bigquery?project={PROJECT_ID}",
    "NHTSA dataset":                  "https://console.cloud.google.com/bigquery?p=bigquery-public-data&d=nhtsa_traffic_fatalities&page=dataset",
    "Census ACS dataset":             "https://console.cloud.google.com/bigquery?p=bigquery-public-data&d=census_bureau_acs&page=dataset",
    "IPEDS dataset (find Morgan!)":   "https://console.cloud.google.com/bigquery?p=bigquery-public-data&d=nces_ipeds&page=dataset",
    "Cloud Storage browser":          f"https://console.cloud.google.com/storage/browser?project={PROJECT_ID}",
    "Model Garden":                   f"https://console.cloud.google.com/vertex-ai/model-garden?project={PROJECT_ID}",
    "Gemma 3 model card":             f"https://console.cloud.google.com/vertex-ai/publishers/google/model-garden/gemma3?project={PROJECT_ID}",
    "Vertex AI endpoints":            f"https://console.cloud.google.com/vertex-ai/online-prediction/endpoints?project={PROJECT_ID}",
    "Vertex AI Studio (Gemini)":      f"https://console.cloud.google.com/vertex-ai/studio?project={PROJECT_ID}",
    "Quotas":                         f"https://console.cloud.google.com/iam-admin/quotas?project={PROJECT_ID}",
    "Billing reports":                f"https://console.cloud.google.com/billing?project={PROJECT_ID}",
}
width = max(len(k) for k in links)
for name, url in links.items():
    print(f"{name:<{width}}  {url}")


### Ideas for *your* research, starting tomorrow
- Swap section 3's SQL for a dataset in your field — [hundreds available](https://console.cloud.google.com/bigquery?p=bigquery-public-data) (air quality, genomics, satellite imagery, patents…)
- Pull a bigger Gemma (`ollama pull gemma3:27b` needs 17 GB of VRAM — your H100 has 80) or browse [Model Garden](https://console.cloud.google.com/vertex-ai/model-garden) for vision, speech, and biology models
- Put your lab's data in the bucket and point the section 5 pipeline at it
- Need more than one GPU? `a3-highgpu` shapes go up to 8× H100 — ask the Google Cloud team at today's session

Thanks for coming to researcher day.
